In [1]:
import csv
import json
import statistics
from pathlib import Path
from typing import Any

# ---------------------------------------------------------------------
# Inputs
# ---------------------------------------------------------------------

USED_QUBITS = [
    7, 17, 25, 26, 27, 28, 29, 37, 38, 43, 44, 45, 46, 47, 48, 49, 50, 51,
    56, 57, 58, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 77, 78, 79, 85,
    86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 97, 98, 99, 107, 108, 109, 110,
    111, 112, 113, 114, 115, 118, 119, 129, 130, 131, 132, 133
]
USED_QUBIT_SET = set(USED_QUBITS)

BACKEND_FILES = {
    "Aachen": "aachen.json",
    "Marrakesh": "marrakesh.json",
    "Pittsburgh": "pittsburgh.json",
}

# If your files are elsewhere, change BASE_DIR.
BASE_DIR = Path(".")

# Output directory for extracted calibration data.
OUT_DIR = Path("derived")


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def load_backend_payload(path: Path) -> dict[str, Any]:
    """
    Load one raw backend-calibration JSON file.
    """
    with open(path, "r") as f:
        data = json.load(f)

    if not isinstance(data, dict) or not data:
        raise ValueError(f"{path} does not contain a non-empty JSON object.")

    return next(iter(data.values()))


def param_dict(param_list: list[dict[str, Any]]) -> dict[str, Any]:
    """
    Convert an IBM-style list of parameter dictionaries into a name -> value map.
    """
    return {entry["name"]: entry["value"] for entry in param_list}


def median_min_max(values: list[float]) -> tuple[float, float, float]:
    """
    Return median, min, and max after dropping None values.
    """
    vals = sorted(float(v) for v in values if v is not None)

    if not vals:
        raise ValueError("No values found for a requested metric.")

    return statistics.median(vals), min(vals), max(vals)


def validate_payload(payload: dict[str, Any], backend_label: str) -> None:
    """
    Basic checks that the raw calibration payload has the expected structure.
    """
    required_fields = ["backend_name", "qubits", "gates"]
    missing = [field for field in required_fields if field not in payload]

    if missing:
        raise KeyError(
            f"{backend_label}: backend payload is missing required fields: {missing}"
        )

    n_qubits_backend = len(payload["qubits"])
    invalid_qubits = [q for q in USED_QUBITS if q >= n_qubits_backend]

    if invalid_qubits:
        raise ValueError(
            f"{backend_label}: USED_QUBITS contains indices outside the backend "
            f"range 0..{n_qubits_backend - 1}: {invalid_qubits}"
        )


def write_csv(path: Path, rows: list[dict[str, Any]]) -> None:
    """
    Write a list of dictionaries to CSV.

    The field order is taken from the first row.
    """
    if not rows:
        raise ValueError(f"No rows to write for {path}")

    path.parent.mkdir(parents=True, exist_ok=True)

    fieldnames = list(rows[0].keys())

    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def write_json(path: Path, obj: Any) -> None:
    """
    Write an object as pretty-printed JSON.
    """
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


# ---------------------------------------------------------------------
# Extraction functions
# ---------------------------------------------------------------------

def extract_used_qubit_calibrations(
    payload: dict[str, Any],
) -> list[dict[str, Any]]:
    """
    Extract one row per used qubit.
    """
    rows = []
    qubits = payload["qubits"]

    for q in USED_QUBITS:
        qd = param_dict(qubits[q])

        rows.append({
            "backend_name": payload.get("backend_name"),
            "last_update_date": payload.get("last_update_date"),
            "qubit": q,

            "T1_us": qd.get("T1"),
            "T2_us": qd.get("T2"),
            "readout_error": qd.get("readout_error"),
            "prob_meas0_prep1": qd.get("prob_meas0_prep1"),
            "prob_meas1_prep0": qd.get("prob_meas1_prep0"),
            "readout_length_ns": qd.get("readout_length"),
        })

    return rows


def extract_used_gate_calibrations(
    payload: dict[str, Any],
) -> list[dict[str, Any]]:
    """
    Extract all gate calibrations whose operands lie entirely in USED_QUBITS.
    """
    rows = []

    for gate in payload["gates"]:
        gate_name = gate.get("gate")
        gate_qubits = gate.get("qubits", [])

        if not all(q in USED_QUBIT_SET for q in gate_qubits):
            continue

        pd = param_dict(gate.get("parameters", []))

        rows.append({
            "backend_name": payload.get("backend_name"),
            "last_update_date": payload.get("last_update_date"),

            "gate": gate_name,
            "qubits": "-".join(map(str, gate_qubits)),
            "qubit_0": gate_qubits[0] if len(gate_qubits) > 0 else None,
            "qubit_1": gate_qubits[1] if len(gate_qubits) > 1 else None,
            "n_qubits": len(gate_qubits),

            "gate_error": pd.get("gate_error"),
            "gate_length_ns": pd.get("gate_length"),
        })

    return rows


def extract_summary_stats(payload: dict[str, Any]) -> dict[str, Any]:
    """
    Extract compact median/min/max summary statistics.
    """
    qubits = payload["qubits"]
    gates = payload["gates"]

    # -------------------------
    # Qubit-level properties
    # -------------------------
    t1_vals = []
    t2_vals = []
    ro_vals = []

    for q in USED_QUBITS:
        qd = param_dict(qubits[q])
        t1_vals.append(qd.get("T1"))
        t2_vals.append(qd.get("T2"))
        ro_vals.append(qd.get("readout_error"))

    # -------------------------
    # Gate-level properties
    #
    # We use:
    #   - sx as representative 1Q gate
    #   - cz as representative 2Q gate
    #
    # For CZ, backend properties sometimes contain both directions [a,b] and
    # [b,a]; we deduplicate those into undirected couplers for the compact
    # summary.
    #
    # Sentinel gate_error >= 1 entries are excluded from the compact summary.
    # -------------------------
    sx_err_vals = []
    sx_len_vals = []

    cz_err_vals = []
    cz_len_vals = []
    seen_edges = set()

    for gate in gates:
        gate_name = gate.get("gate")
        gate_qubits = gate.get("qubits", [])
        pd = param_dict(gate.get("parameters", []))

        # Representative 1Q gate: sx
        if gate_name == "sx" and len(gate_qubits) == 1:
            q = gate_qubits[0]

            if q in USED_QUBIT_SET:
                if pd.get("gate_error") is not None:
                    sx_err_vals.append(pd["gate_error"])
                if pd.get("gate_length") is not None:
                    sx_len_vals.append(pd["gate_length"])

        # Representative 2Q gate: cz
        elif gate_name == "cz" and len(gate_qubits) == 2:
            q0, q1 = gate_qubits

            if q0 in USED_QUBIT_SET and q1 in USED_QUBIT_SET:
                edge = tuple(sorted((q0, q1)))

                if edge in seen_edges:
                    continue

                seen_edges.add(edge)

                gate_error = pd.get("gate_error")
                gate_length = pd.get("gate_length")

                if gate_error is None or float(gate_error) >= 1.0:
                    continue

                cz_err_vals.append(gate_error)

                if gate_length is not None:
                    cz_len_vals.append(gate_length)

    return {
        "backend_name": payload.get("backend_name"),
        "last_update_date": payload.get("last_update_date"),

        "T1_us": median_min_max(t1_vals),
        "T2_us": median_min_max(t2_vals),
        "readout_err": median_min_max(ro_vals),

        "oneq_err": median_min_max(sx_err_vals),
        "twoq_err": median_min_max(cz_err_vals),
        "oneq_len_ns": median_min_max(sx_len_vals),
        "twoq_len_ns": median_min_max(cz_len_vals),

        "n_qubits": len(USED_QUBITS),
        "n_couplers": len(cz_err_vals),
    }


def flatten_summary_stats(
    stats: dict[str, Any],
    backend_label: str,
    source_file: str,
) -> dict[str, Any]:
    """
    Convert nested median/min/max tuples into a flat CSV-friendly row.
    """
    return {
        "backend_label": backend_label,
        "backend_name": stats["backend_name"],
        "source_file": source_file,
        "last_update_date": stats.get("last_update_date"),
        "n_used_qubits": stats["n_qubits"],
        "n_used_cz_couplers": stats["n_couplers"],

        "T1_us_median": stats["T1_us"][0],
        "T1_us_min": stats["T1_us"][1],
        "T1_us_max": stats["T1_us"][2],

        "T2_us_median": stats["T2_us"][0],
        "T2_us_min": stats["T2_us"][1],
        "T2_us_max": stats["T2_us"][2],

        "readout_error_median": stats["readout_err"][0],
        "readout_error_min": stats["readout_err"][1],
        "readout_error_max": stats["readout_err"][2],

        "sx_error_median": stats["oneq_err"][0],
        "sx_error_min": stats["oneq_err"][1],
        "sx_error_max": stats["oneq_err"][2],

        "cz_error_median": stats["twoq_err"][0],
        "cz_error_min": stats["twoq_err"][1],
        "cz_error_max": stats["twoq_err"][2],

        "sx_length_ns_median": stats["oneq_len_ns"][0],
        "sx_length_ns_min": stats["oneq_len_ns"][1],
        "sx_length_ns_max": stats["oneq_len_ns"][2],

        "cz_length_ns_median": stats["twoq_len_ns"][0],
        "cz_length_ns_min": stats["twoq_len_ns"][1],
        "cz_length_ns_max": stats["twoq_len_ns"][2],
    }


def build_manifest() -> dict[str, Any]:
    """
    Repository metadata explaining what the extracted files contain.
    """
    return {
        "description": (
            "Raw calibration files and analysis outputs for the hardware "
            "calibration data associated with the surface-code experiments."
        ),
        "used_qubits": USED_QUBITS,
        "backend_files": BACKEND_FILES,
        "outputs": {
            "used_qubit_calibrations.csv": (
                "One row per used qubit per backend, including T1, T2, "
                "readout error, assignment probabilities, and readout length."
            ),
            "used_gate_calibrations.csv": (
                "One row per calibrated gate whose operands lie entirely in the "
                "used-qubit set, including gate error and gate duration when present."
            ),
            "summary_stats.csv": (
                "Compact median/min/max analysis utility over the used qubits and "
                "representative sx/cz gates."
            ),
            "summary_stats.json": (
                "JSON version of the compact summary statistics."
            ),
            "manifest.json": (
                "Description of inputs, outputs, and conventions used by this script."
            ),
        },
        "summary_conventions": {
            "qubit_statistics": "median, min, and max over USED_QUBITS",
            "representative_one_qubit_gate": "sx",
            "representative_two_qubit_gate": "cz",
            "cz_duplicate_handling": (
                "For compact summary statistics, CZ entries are deduplicated into "
                "undirected couplers by sorting the two qubit indices."
            ),
            "sentinel_gate_errors": (
                "Gate entries with gate_error >= 1 are excluded from compact "
                "summary statistics."
            ),
        },
        "notes": {
            "qubit_csv_column_policy": (
                "frequency and anharmonicity are not included because they are not "
                "present in the supplied qubit-level calibration records."
            ),
            "raw_csv_column_policy": (
                "backend_label and source_file are intentionally omitted from "
                "used_qubit_calibrations.csv and used_gate_calibrations.csv."
            ),
        },
    }


# ---------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------

def main() -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    all_qubit_rows = []
    all_gate_rows = []
    summary_rows = []

    for backend_label, filename in BACKEND_FILES.items():
        path = BASE_DIR / filename
        payload = load_backend_payload(path)
        validate_payload(payload, backend_label)

        all_qubit_rows.extend(
            extract_used_qubit_calibrations(payload=payload)
        )

        all_gate_rows.extend(
            extract_used_gate_calibrations(payload=payload)
        )

        stats = extract_summary_stats(payload)
        summary_rows.append(
            flatten_summary_stats(
                stats=stats,
                backend_label=backend_label,
                source_file=filename,
            )
        )

    write_csv(OUT_DIR / "used_qubit_calibrations.csv", all_qubit_rows)
    write_csv(OUT_DIR / "used_gate_calibrations.csv", all_gate_rows)
    write_csv(OUT_DIR / "summary_stats.csv", summary_rows)
    write_json(OUT_DIR / "summary_stats.json", summary_rows)
    write_json(OUT_DIR / "manifest.json", build_manifest())

    print(f"Wrote {OUT_DIR / 'used_qubit_calibrations.csv'}")
    print(f"Wrote {OUT_DIR / 'used_gate_calibrations.csv'}")
    print(f"Wrote {OUT_DIR / 'summary_stats.csv'}")
    print(f"Wrote {OUT_DIR / 'summary_stats.json'}")
    print(f"Wrote {OUT_DIR / 'manifest.json'}")


if __name__ == "__main__":
    main()

Wrote derived/used_qubit_calibrations.csv
Wrote derived/used_gate_calibrations.csv
Wrote derived/summary_stats.csv
Wrote derived/summary_stats.json
Wrote derived/manifest.json
